In [1]:
import json
import os
import re
import warnings
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")

# System Configuration & Global Seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ==========================================
# 1. PREPROCESSING & PII REDACTION
# ==========================================

def redact_pii(text: str) -> str:
    """Redacts email addresses and phone numbers to ensure PII compliance."""
    if not isinstance(text, str):
        return ""
    email_pattern = r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}"
    phone_pattern = r"\b(?:\+?\d{1,3}[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}\b"
    text = re.sub(email_pattern, "[EMAIL_REDACTED]", text)
    text = re.sub(phone_pattern, "[PHONE_REDACTED]", text)
    return text.strip()

# ==========================================
# 2. MAIN EXECUTION PIPELINE
# ==========================================

def run_pipeline(csv_filename="completeSpamAssassin.csv"):
    print("=" * 60)
    print("      LAB 03: EMAIL CLASSIFICATION PIPELINE STARTING       ")
    print("=" * 60)

    # Search for the CSV file in local directory or /content
    possible_paths = [
        csv_filename,
        os.path.join("/content", csv_filename),
        os.path.join(os.getcwd(), csv_filename),
    ]

    target_path = None
    for p in possible_paths:
        if os.path.exists(p):
            target_path = p
            break

    # Load or fallback to sample dataset
    if target_path:
        print(f"\n[Success] Loading dataset from '{target_path}'...")
        df = pd.read_csv(target_path)
    else:
        print(f"\n[Warning] File '{csv_filename}' not found in workspace.")
        print("[Info] Running on sample dataset for demonstration...\n")
        df = pd.DataFrame({
            "Body": [
                "Dear team, please review the attached document for tomorrow's meeting.",
                "URGENT: Win $10,000 cash prize right now! Click here to claim your reward!",
                "Can we reschedule our technical discussion sync to Thursday at 2 PM?",
                "Buy cheap products online now! Guaranteed heavy discount on all items!",
            ] * 15,
            "Label": [0, 1, 0, 1] * 15,
        })

    # Automatically identify text and label columns
    text_col = "Body" if "Body" in df.columns else df.columns[1]
    label_col = "Label" if "Label" in df.columns else df.columns[-1]

    # Clean records and redact PII
    df = df.dropna(subset=[text_col, label_col]).copy()
    if df[label_col].dtype in [np.int64, np.float64, int, float]:
        df[label_col] = df[label_col].map({0: "legitimate", 1: "spam"}).fillna(df[label_col])

    df["clean_text"] = df[text_col].astype(str).apply(redact_pii)

    print("\n--- DATASET AUDIT SUMMARY ---")
    print(f"Total Rows Evaluated:   {len(df)}")
    print(f"Class Distribution:\n{df[label_col].value_counts().to_string()}")
    print("-" * 60)

    # Stratified Train/Test Split (80/20) - FIXED: stratify=df[label_col]
    X_train_raw, X_test_raw, y_train, y_test = train_test_split(
        df["clean_text"],
        df[label_col],
        test_size=0.2,
        stratify=df[label_col],
        random_state=RANDOM_SEED,
    )

    # Feature Pipeline: TF-IDF Vectorizer
    vectorizer = TfidfVectorizer(
        sublinear_tf=True,
        min_df=1,
        max_df=0.98,
        ngram_range=(1, 2),
        max_features=10000,
    )

    X_train_vec = vectorizer.fit_transform(X_train_raw)
    X_test_vec = vectorizer.transform(X_test_raw)

    # 5-Fold Stratified Cross-Validation
    print("\n--- 1. 5-FOLD CROSS-VALIDATION RESULTS ---")
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

    models = {
        "Dummy Majority": DummyClassifier(strategy="most_frequent"),
        "MultinomialNB": MultinomialNB(alpha=1.0),
        "ComplementNB": ComplementNB(alpha=1.0),
        "Logistic Regression": LogisticRegression(
            C=1.0, class_weight="balanced", random_state=RANDOM_SEED, max_iter=1000
        ),
        "LinearSVC": LinearSVC(
            C=1.0, class_weight="balanced", random_state=RANDOM_SEED
        ),
    }

    cv_results = []
    scoring_metrics = ["accuracy", "f1_macro", "f1_weighted"]

    for name, model in models.items():
        scores = cross_validate(
            model,
            X_train_vec,
            y_train,
            cv=cv,
            scoring=scoring_metrics,
            return_train_score=False,
        )

        cv_results.append(
            {
                "Model": name,
                "Accuracy Mean ± SD": f"{scores['test_accuracy'].mean():.4f} ± {scores['test_accuracy'].std():.4f}",
                "Macro F1 Mean ± SD": f"{scores['test_f1_macro'].mean():.4f} ± {scores['test_f1_macro'].std():.4f}",
                "Weighted F1 Mean": f"{scores['test_f1_weighted'].mean():.4f}",
            }
        )

    cv_df = pd.DataFrame(cv_results)
    print(cv_df.to_string(index=False))

    # Champion Model Evaluation (LinearSVC)
    print("\n--- 2. CHAMPION MODEL (LinearSVC) LOCKED TEST SET EVALUATION ---")
    champion = LinearSVC(
        C=1.0, class_weight="balanced", random_state=RANDOM_SEED
    )
    champion.fit(X_train_vec, y_train)
    y_pred = champion.predict(X_test_vec)

    print(f"Test Accuracy:    {accuracy_score(y_test, y_pred):.4f}")
    print(f"Macro Precision:  {precision_score(y_test, y_pred, average='macro'):.4f}")
    print(f"Macro Recall:     {recall_score(y_test, y_pred, average='macro'):.4f}")
    print(f"Macro F1:         {f1_score(y_test, y_pred, average='macro'):.4f}")
    print(f"Weighted F1:      {f1_score(y_test, y_pred, average='weighted'):.4f}\n")

    print("Per-Class Classification Report:\n")
    print(classification_report(y_test, y_pred))

    # Sample Draft Output Structure
    sample_email = X_test_raw.iloc[0]
    sample_pred = y_pred[0]

    print("--- 3. SAMPLE DRAFT GENERATION ARTIFACT ---")
    draft_record = {
        "predicted_class": sample_pred,
        "draft_generated": False if sample_pred == "spam" else True,
        "mandatory_review": False if sample_pred != "urgent_action" else True,
        "reason": (
            "Draft suppressed: Spam message detected."
            if sample_pred == "spam"
            else "Ready for operational review."
        ),
        "draft_text": (
            None
            if sample_pred == "spam"
            else f"Dear Sender,\n\nThank you for reaching out regarding your '{sample_pred}' request.\nWe have received your email and will follow up shortly.\n\nBest regards,\n[PLACEHOLDER - Support Team]"
        ),
    }
    print(json.dumps(draft_record, indent=4))
    print("=" * 60)

# Run code directly
run_pipeline("completeSpamAssassin.csv")

      LAB 03: EMAIL CLASSIFICATION PIPELINE STARTING       

[Warning] File 'completeSpamAssassin.csv' not found in workspace.
[Info] Running on sample dataset for demonstration...


--- DATASET AUDIT SUMMARY ---
Total Rows Evaluated:   60
Class Distribution:
Label
legitimate    30
spam          30
------------------------------------------------------------

--- 1. 5-FOLD CROSS-VALIDATION RESULTS ---
              Model Accuracy Mean ± SD Macro F1 Mean ± SD Weighted F1 Mean
     Dummy Majority    0.4778 ± 0.0272    0.3231 ± 0.0126           0.3094
      MultinomialNB    1.0000 ± 0.0000    1.0000 ± 0.0000           1.0000
       ComplementNB    1.0000 ± 0.0000    1.0000 ± 0.0000           1.0000
Logistic Regression    1.0000 ± 0.0000    1.0000 ± 0.0000           1.0000
          LinearSVC    1.0000 ± 0.0000    1.0000 ± 0.0000           1.0000

--- 2. CHAMPION MODEL (LinearSVC) LOCKED TEST SET EVALUATION ---
Test Accuracy:    1.0000
Macro Precision:  1.0000
Macro Recall:     1.0000
Macr